# v2: 多频率融合 Transformer 端到端量价时序预测 (BigAlpha 2026)

相对 v1 (`Transformer_predict.ipynb`) 的改进:
1. **多频率融合**: 同时使用 1m / 5m / 15m / 30m 四张原始 K 线表, 每频率一个编码分支,
   融合后预测次日收益, 同时捕捉日内微观结构与更长周期趋势;
2. **注意力池化**: 用可学习 query 对各时间步加权, 替代 mean 池化;
3. **训练技巧**: AdamW + OneCycle 学习率 (warmup + cosine) + 梯度裁剪.

合规要点:
- 每频率 11 个核心原始字段 × 4 频率 = 44 个输入 (≤100), 无任何衍生/交互特征;
- 预处理仅 pre_close 归一化 / log1p / 按字段 StandardScaler / 缺失填充;
- 每频率窗口 = 1 个交易日 (1m=240bar, 5m=48, 15m=16, 30m=8) ≤ 240 交易日;
- 参数量约 113 万 (10万~1亿), 无外部预训练权重, 从零训练.

运行逻辑: 若同目录存在 `transformer_model_v2.json` (预训练权重) 则直接加载快速推理;
否则在写死的训练区间上在线训练 (兜底).

In [ ]:
def main(datasources, start_date, end_date):
    """v2: 多频率融合 Transformer 端到端推理 (BigAlpha 2026 端到端量价时序预测).

    赛制约定: 平台只替换 datasources / start_date / end_date, 其中
    start_date~end_date 为【测试集区间】。训练区间写死(TRAIN_START/END),
    用样本外的测试区间做预测, 输出每日分数 ['date','instrument','score']。
    切勿用传入的 start_date/end_date 训练(数据泄漏, 会被审查)。

    数据表约定: 训练读【写死的开发数据表】(含历史区间), 推理读【平台注入的】
    datasources (公榜/私榜会换成对应后缀的物理表)。两者绝不能混用。
    """
    import os
    import json
    import math
    import time
    import numpy as np
    import pandas as pd
    import dai
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    import structlog

    logger = structlog.get_logger()

    # ---------- 配置 (写死, 不随平台入参变化) ----------
    TRAIN_TABLES = {   # 训练用【写死的开发数据表】, 含历史训练区间
        "bar1m": "bigalpha_2026_stock_bar1m",
        "bar5m": "bigalpha_2026_stock_bar5m",
        "bar15m": "bigalpha_2026_stock_bar15m",
        "bar30m": "bigalpha_2026_stock_bar30m",
    }
    TRAIN_START, TRAIN_END = "2022-01-01", "2023-12-31 23:59:59"
    EPOCHS, BATCH, LR, SEED = 6, 256, 1e-3, 42
    WARMUP_RATIO = 0.1            # OneCycle warmup 占比
    WEIGHT_DECAY = 1e-4
    MAX_TRAIN_INSTRUMENTS = 200   # 训练标的数上限 (可按资源上调)
    LOOKBACK_BUF_DAYS = 10        # 取数缓冲, 用于凑回看窗口
    QUERY_CHUNK = 100             # 每批查询的标的数 (分块读取, 降低峰值内存)

    # 每个频率的核心原始字段 (11 个; 4 频率共 44 个输入, ≤100 合规)
    BASE_PRICE_COLS = ["open", "high", "low", "close", "bid_price1", "ask_price1"]
    BASE_LOG_COLS = ["volume", "amount", "bid_volume1", "ask_volume1"]
    BASE_RAW_COLS = ["pre_close"]
    BASE_FEATURE_COLS = BASE_PRICE_COLS + BASE_LOG_COLS + BASE_RAW_COLS

    # 频率配置: key 对应 datasources, seq_len 为该频率"1 个交易日"的 bar 数
    FREQ_CFGS = [
        dict(key="bar1m",  seq_len=240, label_source=True),
        dict(key="bar5m",  seq_len=48),
        dict(key="bar15m", seq_len=16),
        dict(key="bar30m", seq_len=8),
    ]
    MODEL_CFG = dict(d_model=128, nhead=8, nlayers=2, dim_ff=256, dropout=0.1)

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    logger.info("运行设备", device=str(DEVICE), n_freq=len(FREQ_CFGS))

    # ---------- 模型: 多频率分支 + 注意力池化 + 融合回归头 ----------
    class FreqBranch(nn.Module):
        def __init__(self, n_feat, seq_len, d_model=128, nhead=8, nlayers=2,
                     dim_ff=256, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.proj = nn.Linear(n_feat, d_model)
            self.pos = nn.Parameter(torch.zeros(1, seq_len, d_model))
            self.in_drop = nn.Dropout(dropout)
            layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout,
                                               batch_first=True, activation="gelu")
            self.encoder = nn.TransformerEncoder(layer, nlayers)
            self.pool_query = nn.Parameter(torch.randn(d_model) * 0.02)

        def forward(self, x):                                # (B,L,F) -> (B,d_model)
            h = self.in_drop(self.proj(x) + self.pos)
            h = self.encoder(h)
            scores = torch.matmul(h, self.pool_query) / math.sqrt(self.d_model)
            w = torch.softmax(scores, dim=1)
            return (h * w.unsqueeze(-1)).sum(dim=1)

    class MultiFreqTransformer(nn.Module):
        def __init__(self, freq_cfgs, d_model=128, nhead=8, nlayers=2,
                     dim_ff=256, dropout=0.1):
            super().__init__()
            self.branches = nn.ModuleList([
                FreqBranch(cfg["n_feat"], cfg["seq_len"], d_model, nhead, nlayers,
                           dim_ff, dropout) for cfg in freq_cfgs
            ])
            n_freq = len(freq_cfgs)
            self.head = nn.Sequential(
                nn.Linear(d_model * n_freq, d_model),   # 融合: 拼接 -> d_model
                nn.LayerNorm(d_model),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model, 1),
            )

        def forward(self, xs):                             # list[(B,L,F)*4] -> (B,)
            hs = [branch(x) for branch, x in zip(self.branches, xs)]
            h = torch.cat(hs, dim=-1)
            return self.head(h).squeeze(-1)

    # ---------- 工具 ----------
    def pool(table, sd, ed):
        df = dai.query(f"SELECT DISTINCT instrument FROM {table}",
                       filters={"date": [sd, ed]}).df()
        return df["instrument"].tolist()

    def resolve_available_cols(table, sd, ed):
        probe_sd = pd.to_datetime(sd)
        probe_ed = min(pd.to_datetime(ed), probe_sd + pd.Timedelta(days=7))
        probe = dai.query(f"SELECT * FROM {table}",
                          filters={"date": [probe_sd.strftime("%Y-%m-%d"),
                                            probe_ed.strftime("%Y-%m-%d 23:59:59")]}).df()
        cols = [c for c in BASE_FEATURE_COLS if c in probe.columns]
        del probe
        return cols

    def preprocess_features(sub, price_cols, log_cols, has_pc):
        pc = sub["pre_close"].to_numpy(np.float32) if has_pc else None
        for c in price_cols:
            arr = sub[c].to_numpy(np.float32)
            if pc is not None:
                denom = np.where(np.isfinite(pc) & (pc > 0), pc, np.float32(1.0))
                arr = arr / denom
            sub[c] = arr
        for c in log_cols:
            sub[c] = np.log1p(sub[c].clip(lower=0).to_numpy(np.float32))
        keep = price_cols + log_cols + (["pre_close"] if has_pc else [])
        sub[keep] = sub[keep].ffill().fillna(0.0)
        return sub

    def build_dataset(table, sd, ed, mode, instruments, feat_cols, seq_len,
                      stats=None, need_label=False):
        t0 = time.time()
        buf = (pd.to_datetime(sd) - pd.Timedelta(days=LOOKBACK_BUF_DAYS)).strftime("%Y-%m-%d")
        sd_ts, ed_ts = pd.to_datetime(sd), pd.to_datetime(ed)
        price_cols = [c for c in BASE_PRICE_COLS if c in feat_cols]
        log_cols = [c for c in BASE_LOG_COLS if c in feat_cols]
        has_pc = "pre_close" in feat_cols
        wins, ys, keys = [], [], []
        for i in range(0, len(instruments), QUERY_CHUNK):
            batch = instruments[i:i + QUERY_CHUNK]
            sql = (f"SELECT date, instrument, {', '.join(feat_cols)} FROM {table} "
                   f"ORDER BY instrument, date")
            df = dai.query(sql, filters={"date": [buf, ed], "instrument": batch},
                           compression=True).df()
            df = df[["date", "instrument"] + feat_cols]
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values(["instrument", "date"])
            for ins_, sub in df.groupby("instrument", sort=False):
                raw_close = sub["close"].to_numpy(np.float64) if "close" in sub.columns else None
                day = sub["date"].dt.normalize().to_numpy()
                close_pos = np.flatnonzero(np.append(day[1:] != day[:-1], True))
                close_px = raw_close[close_pos] if raw_close is not None else None
                dates = day[close_pos]
                sub = preprocess_features(sub, price_cols, log_cols, has_pc)
                arr = sub[feat_cols].to_numpy(np.float32)
                for k, p in enumerate(close_pos):
                    d = pd.Timestamp(dates[k])
                    if p + 1 < seq_len or d < sd_ts or d > ed_ts:
                        continue
                    label = None
                    if need_label and close_px is not None and k + 1 < len(close_px) and close_px[k] > 0:
                        r = close_px[k + 1] / close_px[k] - 1.0
                        if np.isfinite(r):
                            label = np.float32(r)
                    if mode == "train" and need_label and label is None:
                        continue
                    wins.append(arr[p - seq_len + 1: p + 1])
                    ys.append(label if label is not None else np.float32(0.0))
                    keys.append((d, ins_))
            del df
        if not keys:
            raise RuntimeError(f"build_dataset 无样本 (mode={mode}, table={table}, {sd}~{ed})")
        X = np.stack(wins).astype(np.float32)
        if stats is None:
            flat = X.reshape(-1, len(feat_cols))
            stats = (flat.mean(0).astype(np.float32), flat.std(0).astype(np.float32) + 1e-6)
        m_, s_ = stats
        X = ((X - m_) / s_).astype(np.float32)
        logger.info(f"{mode} 集构建完成", table=table, samples=len(keys),
                    elapsed=round(time.time() - t0, 2))
        keys_df = pd.DataFrame(keys, columns=["date", "instrument"])
        if mode == "train":
            return X, np.array(ys, np.float32), keys_df, stats
        return X, None, keys_df, stats

    def align_frequencies(per_freq):
        base = per_freq[0]
        base_keys = list(map(tuple, base["keys"][["date", "instrument"]].to_numpy()))
        common = set(base_keys)
        for p in per_freq[1:]:
            ks = list(map(tuple, p["keys"][["date", "instrument"]].to_numpy()))
            common &= set(ks)
        common = sorted(common)
        idxs = []
        for p in per_freq:
            kmap = {k: i for i, k in enumerate(map(tuple, p["keys"][["date", "instrument"]].to_numpy()))}
            idxs.append(np.array([kmap[k] for k in common], dtype=np.int64))
        Xs = [p["X"][idx] for p, idx in zip(per_freq, idxs)]
        y = base["y"][idxs[0]] if base.get("y") is not None else None
        keys_df = pd.DataFrame(list(common), columns=["date", "instrument"])
        return Xs, y, keys_df

    def load_model(model_path):
        with open(model_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        sd = {}
        for k, meta in payload["state_dict"].items():
            t = torch.tensor(meta["data"], dtype=getattr(torch, meta["dtype"]))
            sd[k] = t.reshape(meta["shape"])
        ckpt = {k: v for k, v in payload.items() if k != "state_dict"}
        ckpt["state_dict"] = sd
        return ckpt

    def predict(model, Xs, batch, device):
        model.eval()
        preds = []
        ts = [torch.from_numpy(x) for x in Xs]
        n = ts[0].shape[0]
        with torch.no_grad():
            for j in range(0, n, batch):
                xs = [t[j:j + batch].to(device) for t in ts]
                preds.append(model(xs).cpu().numpy())
        return np.concatenate(preds).astype(np.float64)

    def run_inference(model, tables, sd, ed, ins_list, freq_cfgs, feat_cols_list,
                      stats_list, batch, device):
        all_rows = []
        for i in range(0, len(ins_list), QUERY_CHUNK):
            batch_ins = ins_list[i:i + QUERY_CHUNK]
            per_freq = []
            for cfg, table, feats, st in zip(freq_cfgs, tables, feat_cols_list, stats_list):
                X, _, keys_df, _ = build_dataset(
                    table, sd, ed, "infer", batch_ins, feats, cfg["seq_len"],
                    stats=st, need_label=False)
                per_freq.append(dict(X=X, y=None, keys=keys_df))
            Xs, _, keys_df = align_frequencies(per_freq)
            if len(keys_df) == 0:
                continue
            keys_df["score"] = predict(model, Xs, batch, device)
            all_rows.append(keys_df)
            del Xs
        if all_rows:
            return pd.concat(all_rows, ignore_index=True)
        return pd.DataFrame(columns=["date", "instrument", "score"])

    # ---------- 权重: 优先加载提交的预训练模型, 否则在线训练 ----------
    model = None
    stats_list = None
    feat_cols_list = None
    freq_cfgs = None
    model_path = "transformer_model_v2.json"
    if os.path.exists(model_path):
        try:
            ckpt = load_model(model_path)
            model = MultiFreqTransformer(**ckpt["model_cfg"])
            model.load_state_dict(ckpt["state_dict"])
            model = model.to(DEVICE).eval()
            freq_cfgs = list(ckpt["freq_cfgs"])
            feat_cols_list = [list(f) for f in ckpt["feature_cols"]]
            stats_list = [(np.asarray(p[0], np.float32), np.asarray(p[1], np.float32))
                          for p in ckpt["stats"]]
            logger.info("已加载预训练权重", path=model_path,
                        n_params=sum(p.numel() for p in model.parameters()))
        except Exception as e:
            logger.warning("预训练权重加载失败, 转为在线训练", error=str(e))
            model = None

    if model is None:
        # ---------- 在线训练 (写死训练区间 + 写死开发数据表, 从零训练) ----------
        train_ins = pool(TRAIN_TABLES["bar1m"], TRAIN_START, TRAIN_END)[:MAX_TRAIN_INSTRUMENTS]
        per_freq, feat_cols_list = [], []
        for cfg in FREQ_CFGS:
            table = TRAIN_TABLES[cfg["key"]]
            feat_cols = resolve_available_cols(table, TRAIN_START, TRAIN_END)
            logger.info("构建训练集", table=table, start=TRAIN_START, end=TRAIN_END,
                        n_feat=len(feat_cols))
            X, y, keys_df, stats = build_dataset(
                table, TRAIN_START, TRAIN_END, "train", train_ins, feat_cols,
                cfg["seq_len"], stats=None, need_label=cfg.get("label_source", False))
            per_freq.append(dict(X=X, y=y, keys=keys_df, stats=stats))
            feat_cols_list.append(feat_cols)
        Xs, y, _ = align_frequencies(per_freq)
        lo, hi = np.percentile(y, [1, 99])
        y = np.clip(y, lo, hi)
        stats_list = [p["stats"] for p in per_freq]
        freq_cfgs = [
            dict(key=cfg["key"], seq_len=cfg["seq_len"], n_feat=len(feats))
            for cfg, feats in zip(FREQ_CFGS, feat_cols_list)
        ]
        model_cfg = dict(freq_cfgs=freq_cfgs, **MODEL_CFG)
        model = MultiFreqTransformer(**model_cfg).to(DEVICE)
        logger.info("可训练参数量", n_params=sum(p.numel() for p in model.parameters()))
        loader = DataLoader(TensorDataset(*[torch.from_numpy(x) for x in Xs],
                                          torch.from_numpy(y)),
                            batch_size=BATCH, shuffle=True, pin_memory=(DEVICE.type == "cuda"))
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        total_steps = max(len(loader) * EPOCHS, 1)
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=LR, total_steps=total_steps,
            pct_start=WARMUP_RATIO, anneal_strategy="cos")
        loss_fn = nn.MSELoss()
        model.train()
        for ep in range(EPOCHS):
            t, tot, nb = time.time(), 0.0, 0
            for batch_data in loader:
                xs = [xb.to(DEVICE, non_blocking=True) for xb in batch_data[:-1]]
                yb = batch_data[-1].to(DEVICE, non_blocking=True)
                opt.zero_grad()
                loss = loss_fn(model(xs), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                sched.step()
                tot += loss.item()
                nb += 1
            logger.info("epoch 完成", epoch=ep + 1, mse=round(tot / max(nb, 1), 8),
                        elapsed=round(time.time() - t, 2))
        model.eval()

    # ---------- 推理 (样本外测试区间, 用平台注入表) ----------
    logger.info("构建测试集并预测", start=str(start_date), end=str(end_date))
    infer_tables = [datasources[cfg["key"]] for cfg in freq_cfgs]
    infer_feats_list = []
    for cfg, feats in zip(freq_cfgs, feat_cols_list):
        avail = resolve_available_cols(infer_tables[freq_cfgs.index(cfg)], start_date, end_date)
        infer_feats_list.append([c for c in feats if c in avail])
    ins_list = pool(infer_tables[0], start_date, end_date)
    idx_df = run_inference(model, infer_tables, start_date, end_date, ins_list,
                           freq_cfgs, infer_feats_list, stats_list, BATCH, DEVICE)

    # ---------- 对齐中证 1000 + 规范输出 ----------
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    result = (pd.merge(idx_df, stk, on=["date", "instrument"], how="inner")
                .replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
                .drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
                .reset_index(drop=True))
    logger.info("分数构建完成", rows=len(result), days=result["date"].nunique(),
                instruments=result["instrument"].nunique())
    return result

In [ ]:
if __name__ == "__main__":
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()

    # 使用 1m/5m/15m/30m 四张 K 线表作为输入数据
    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "bar5m": "bigalpha_2026_stock_bar5m",
        "bar15m": "bigalpha_2026_stock_bar15m",
        "bar30m": "bigalpha_2026_stock_bar30m",
    }

    # 本地用一小段区间模拟「平台注入的测试集区间」(训练区间已在 main 内写死)
    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    logger.info("计算分数", start=start_date, end=end_date)
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())

    # 评估系统: 分数经风格剔除后等价于每日单因子, show=True 画绩效图 (IC / 分组 / 压力期)
    logger.info("开始评估分数")
    result = M.bigalpha_eval._latest(
        factor_data=score_data,
        show=True,
    )